#  Spark Architecture & Efficient Data Processing

**Objective:** Understand Spark architecture and perform efficient data processing using transformations, filtering, schema handling, and optimized file formats.

**Dataset:** `orders_data.csv` 

This notebook works through each step of the objective:
1. Spark architecture (Driver, Cluster Manager, Executors) & execution modes
2. Lazy Evaluation and the DAG / Lineage Graph
3. Reading files (CSV, Parquet) with proper schema handling
4. Filtering and selecting columns
5. Modifying DataFrames (rename, cast, add columns)
6. Transformations vs Actions
7. Wide transformations & performance (Shuffle, Predicate Pushdown)
8. CSV vs Parquet and performance impact
9. Handling null values and efficient filtering
10. Building a pipeline (read → transform → filter → write)
11. Saving to CSV / Parquet
12. Best practices for large datasets (`show()` over `collect()`)

Each section includes PySpark code, execution results, and brief insights.

---

## 1. Spark Architecture & Execution Modes

**Core components :**

- **Driver** : The `Driver` is the main application process that starts your Spark program. It creates the `SparkSession`, maintains the `SparkContext`, generates the execution plan (DAG), divides the job into stages and tasks, and coordinates their execution. It also collects the results returned by the executors.

- **Cluster Manager** : The `Cluster Manager` is responsible for distributing computing resources such as CPU and memory across the cluster. Based on the driver's request, it launches executors on worker nodes. Spark supports multiple cluster managers, including `Standalone`, `YARN`, `Apache Mesos`, and `Kubernetes`.

- **Executors** : `Executors` are JVM processes running on worker machines. They execute the tasks assigned by the driver, store partitioned data in memory or on disk, and send the processed results back to the driver. Each executor can process multiple tasks simultaneously depending on the number of CPU cores allocated.

**Execution modes :**

- **Client mode** : In `Client Mode`, the driver program runs on the same machine from which the Spark application is submitted, such as a local computer or an edge node. This mode is commonly used for development and interactive analysis. Since the driver remains on the submission machine, the application stops if that machine disconnects or crashes.

- **Cluster mode** : In `Cluster Mode`, the driver is launched within the cluster itself on one of the worker nodes. This makes the application independent of the machine that submitted the job, providing greater reliability and making it the preferred choice for production workloads.

The cell below inspects the live session to show these concepts concretely :-

In [23]:
# Set the HADOOP_HOME environment variable 
import os
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = os.environ["PATH"] + r";C:\hadoop\bin"

In [24]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import (StructType, StructField, StringType,
                               DoubleType, IntegerType)

# creating a spark session
spark = (SparkSession.builder
         .appName("Week6_Spark_Architecture")    # name of the application
         .master("local[*]")    # run locally with all available cores
         .getOrCreate())

# Show only error logs
spark.sparkContext.setLogLevel("ERROR")     

# get SparkContext
sc = spark.sparkContext
print("Spark version :", spark.version)
print("Application name :", sc.appName)
print("Master :", sc.master)

Spark version : 3.5.1
Application name : Week6_Spark_Architecture
Master : local[*]


## 2. Lazy Evaluation & the DAG (Lineage Graph)

Spark follows a **lazy evaluation** approach. Operations such as `filter`, `select`, and `withColumn` are not executed immediately. Instead, Spark records these operations as a **Directed Acyclic Graph (DAG)**, also known as the **lineage**, which represents the sequence of transformations.

Execution begins only when an **action** such as `show()`, `count()`, or `write()` is called. At that point, Spark reviews the complete execution plan, applies optimizations through the Catalyst optimizer, and then processes the data.

**Why this improves performance :** because Spark sees the entire chain before running it, it can combine narrow operations into a single pass over the data, skip columns you never use, and avoid materializing intermediate results.

Below code, we can see the lazy execution and the DAG :

In [25]:
# Build a chain of transformations (executed lazily)
lazy_df = (spark.read.csv("../data/orders_data.csv", header=True, inferSchema=True)
           .filter(col("category") == "Electronics")
           .select("product_id", "price"))

# At this stage, Spark has only created the execution plan (DAG)
print("Transformations defined, but no job has executed yet.")

# The action below is what actually triggers execution
print("Row count (action triggers the DAG):", lazy_df.count())

Transformations defined, but no job has executed yet.
Row count (action triggers the DAG): 1363


In [26]:
# View the logical and physical execution plan
lazy_df.explain(mode="formatted")

== Physical Plan ==
* Project (3)
+- * Filter (2)
   +- Scan csv  (1)


(1) Scan csv 
Output [3]: [product_id#1057, category#1058, price#1059]
Batched: false
Location: InMemoryFileIndex [file:/d:/CEI internship/Week 6/Spark_Assignment/data/orders_data.csv]
PushedFilters: [IsNotNull(category), EqualTo(category,Electronics)]
ReadSchema: struct<product_id:string,category:string,price:double>

(2) Filter [codegen id : 1]
Input [3]: [product_id#1057, category#1058, price#1059]
Condition : (isnotnull(category#1058) AND (category#1058 = Electronics))

(3) Project [codegen id : 1]
Output [2]: [product_id#1057, price#1059]
Input [3]: [product_id#1057, category#1058, price#1059]




## 3. Reading Files with Proper Schema Handling

Reading a CSV with `header` and `inferSchema` :

In [27]:
# Read the CSV file into a DataFrame
df = spark.read.csv("../data/orders_data.csv", header=True, inferSchema=True)

# Display the number of rows and columns in the DataFrame
print("Rows:", df.count(), "| Columns:", len(df.columns))
df.show(7)  

Rows: 10000 | Columns: 14
+---------+----------+---------+-------+----------+-------+---------+-------+-------+--------+----------+--------+--------------+--------+
| order_id|product_id| category|  price|base_price| amount|   status|user_id| region|priority|order_date|quantity|payment_method|old_name|
+---------+----------+---------+-------+----------+-------+---------+-------+-------+--------+----------+--------+--------------+--------+
|ORD100000|     P1072| Clothing|1268.94|   1793.83|2123.09| Refunded|  U3013|  South|  Medium|2025-06-19|       5|   Credit Card|item_366|
|ORD100001|     P1372|    Books|3654.49|   1842.62|4088.83| Refunded|  U2786|  South|  Medium|2024-01-25|       1|    Debit Card|item_357|
|ORD100002|     P1479|     Toys|1626.28|   1942.21|2520.27|Cancelled|   NULL|   East|     Low|2024-12-27|      13|   Credit Card|item_159|
|ORD100003|     P1089|Groceries|3072.87|   1416.06|4873.15|Completed|  U3148|  North|     Low|2025-09-10|      14|           UPI|item_245|
|

In [28]:
# Displaying the DataFrame schema
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- old_name: string (nullable = true)



## 4. Filtering & Selecting Columns

- **Q5** - select `product_id` and `price` where category is 'Electronics' :

In [29]:
electronics = df.filter(col("category") == "Electronics").select("product_id", "price")
print("Electronics rows:", electronics.count())
electronics.show(7)

Electronics rows: 1363
+----------+-------+
|product_id|  price|
+----------+-------+
|     P1311|1940.96|
|     P1546|4310.07|
|     P1519|3484.92|
|     P1323|3401.56|
|     P1950|1413.87|
|     P1224| 209.12|
|     P1339|3726.28|
+----------+-------+
only showing top 7 rows



- **Q8** - filter for status = 'Completed' AND amount > 1000:

In [30]:
completed = df.filter((col("status") == "Completed") & (col("amount") > 1000))
print("Completed orders over 1000:", completed.count())
completed.select("order_id", "status", "amount").show(7)

Completed orders over 1000: 1777
+---------+---------+-------+
| order_id|   status| amount|
+---------+---------+-------+
|ORD100003|Completed|4873.15|
|ORD100004|Completed|1826.08|
|ORD100006|Completed|7758.76|
|ORD100007|Completed|3992.74|
|ORD100009|Completed|3199.83|
|ORD100019|Completed|4742.13|
|ORD100026|Completed|5735.06|
+---------+---------+-------+
only showing top 7 rows



- **Q14** - filter for region = 'North' OR priority = 'High':

In [31]:
north_or_high = df.filter((col("region") == "North") | (col("priority") == "High"))
print("North OR High-priority rows:", north_or_high.count())
north_or_high.select("order_id", "region", "priority").show(7)

North OR High-priority rows: 4662
+---------+-------+--------+
| order_id| region|priority|
+---------+-------+--------+
|ORD100003|  North|     Low|
|ORD100004|  North|  Medium|
|ORD100005|   East|    High|
|ORD100006|Central|    High|
|ORD100010|  North|  Medium|
|ORD100016|  North|  Medium|
|ORD100018|  South|    High|
+---------+-------+--------+
only showing top 7 rows



## 5. Modifying DataFrames - Rename, Add Columns

- **Q6** - rename `old_name` to `new_name`

In [32]:
df_revised = (df
    .withColumnRenamed("old_name", "new_name")
)

df_revised.select("new_name").show(7)

+--------+
|new_name|
+--------+
|item_366|
|item_357|
|item_159|
|item_245|
|item_381|
|item_204|
|item_422|
+--------+
only showing top 7 rows



- **Q10** - add a new column `final_price` = `base_price` × 1.18 (18% tax):

In [33]:
df_taxed = df_revised.withColumn("final_price", F.round(col("base_price") * 1.18, 2))
df_taxed.select("base_price", "final_price").show(7)

+----------+-----------+
|base_price|final_price|
+----------+-----------+
|   1793.83|    2116.72|
|   1842.62|    2174.29|
|   1942.21|    2291.81|
|   1416.06|    1670.95|
|   2881.66|    3400.36|
|   2076.37|    2450.12|
|   2649.05|    3125.88|
+----------+-----------+
only showing top 7 rows



## 6. Transformations vs Actions

- **Transformations** are *lazy*  they define a new DataFrame and return immediately without computing. Examples: `filter()`, `select()`, `withColumn()`, `groupBy()`.

- **Actions** *trigger execution* of the DAG and return a result to the driver (or write output). Examples: `count()`, `show()`, `collect()`, `write()`.

The following example demonstrates how a transformation is created without execution, while an action initiates the processing of the entire execution plan :

In [34]:
# Transformation: returns instantly, no job runs
t = df.filter(col("amount") > 5000).select("order_id", "amount")
print("Transformation defined (lazy) - no job has executed yet.")

# Action: triggers computation
print("Action result :", t.count())

Transformation defined (lazy) - no job has executed yet.
Action result : 3780


## 7. Wide Transformations & Performance (Shuffle, Predicate Pushdown)

- **Narrow transformations** such as `filter()`, `select()`, and `withColumn()` process data within the same partition. Since data does not need to move between partitions, these operations are generally fast and efficient.

- **Wide transformations** such as `groupBy()`, `join()`, and `distinct()` require data from multiple partitions to be reorganized. This process, known as a **shuffle**, transfers data across the cluster and redistributes it based on a key. Because shuffling involves network communication and additional disk I/O, it is one of the most expensive operations in Spark.



**Q7 - Fault tolerance via the Lineage Graph :** 

Spark maintains a **lineage graph** that records the sequence of transformations applied to the data. If a partition is lost due to an executor failure, Spark can rebuild only the missing partition using its lineage instead of recomputing the entire dataset. This makes Spark fault tolerant and efficient.

**Q9 - Predicate Pushdown:** 

When working with columnar file formats such as **Parquet**, Spark can apply **predicate pushdown**. Filter conditions are pushed to the data source so that only the required data is read from disk. By skipping irrelevant row groups, Spark reduces disk I/O, memory usage, and overall execution time.

- The example below uses `groupBy()` and `explain()` to display the execution plan, where the **Exchange** operator indicates that a shuffle is taking place :

In [35]:
(df.groupBy("region")
   .agg(F.count("*").alias("orders"),
        F.round(F.avg("amount"), 2).alias("avg_amount"))
   .explain())

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[region#1120], functions=[count(1), avg(amount#1117)])
   +- Exchange hashpartitioning(region#1120, 200), ENSURE_REQUIREMENTS, [plan_id=1111]
      +- HashAggregate(keys=[region#1120], functions=[partial_count(1), partial_avg(amount#1117)])
         +- FileScan csv [amount#1117,region#1120] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/d:/CEI internship/Week 6/Spark_Assignment/data/orders_data.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<amount:double,region:string>




In [36]:
# The actual grouped result (a wide transformation)
(df.groupBy("region")
   .agg(F.count("*").alias("orders"),
        F.round(F.avg("amount"), 2).alias("avg_amount"))
   .orderBy(F.col("orders").desc())
   .show())

+-------+------+----------+
| region|orders|avg_amount|
+-------+------+----------+
|   West|  2084|   4073.57|
|   East|  2020|   4022.63|
|  South|  1993|   3955.69|
|Central|  1952|   4117.12|
|  North|  1951|   4049.41|
+-------+------+----------+



## 8. CSV vs Parquet - Storage & Performance

**Q4 -> Row-based vs Columnar:**

| | CSV | Parquet |
|---|---|---|
| Layout | **Row-based** - values stored row by row | **Columnar** - values stored column by column |
| Schema | None (inferred/declared each read) | Schema stored in the file |
| Compression | Poor (mixed types per row) | Excellent (similar values grouped per column) |
| Column pruning | Must read whole rows | Reads only needed columns |
| Predicate pushdown | Not supported | Supported (row-group stats) |

**Why Parquet is Preferred :** CSV stores data row by row, making it suitable for data exchange but less efficient for large-scale analytics. Parquet stores data column by column, allowing Spark to read only the required columns and skip irrelevant data during queries. This results in lower disk I/O, reduced memory usage, better compression, and faster query execution, making Parquet the preferred format for modern data engineering workloads.

## 9. Handling Null Values & Efficient Filtering

The dataset has empty `price` values and null `user_id`s. Below we quantify and handle them :

In [37]:
from pyspark.sql.functions import when, count

null_report = df.select([
    count(when(col(c).isNull() | (col(c).cast("string") == ""), c)).alias(c)
    for c in df.columns
])
null_report.show(truncate=False)

+--------+----------+--------+-----+----------+------+------+-------+------+--------+----------+--------+--------------+--------+
|order_id|product_id|category|price|base_price|amount|status|user_id|region|priority|order_date|quantity|payment_method|old_name|
+--------+----------+--------+-----+----------+------+------+-------+------+--------+----------+--------+--------------+--------+
|0       |0         |0       |523  |0         |0     |0     |729    |0     |0       |0         |0       |0             |0       |
+--------+----------+--------+-----+----------+------+------+-------+------+--------+----------+--------+--------------+--------+



- **Q12** - filter out any rows where `user_id` is null :

In [38]:
# Filter out rows with null/empty user_id 
df_valid = df.filter(col("user_id").isNotNull() & (col("user_id") != ""))
print("Rows before:", df.count(), "| after removing null user_id:", df_valid.count())

Rows before: 10000 | after removing null user_id: 9271


## 10 & 11. Build a Pipeline (read → transform → filter → write) and Save

A complete pipeline that reads the data, transforms it (rename, cast, add tax column), filters out invalid rows, and writes the result to **both** Parquet and CSV.

In [39]:
from pyspark.sql.types import (StructType, StructField, StringType,
                               DoubleType, IntegerType)

schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("category",       StringType(),  True),
    StructField("price",          StringType(),  True),   # kept as string, cast later in pipeline
    StructField("base_price",     DoubleType(),  True),
    StructField("amount",         DoubleType(),  True),
    StructField("status",         StringType(),  True),
    StructField("user_id",        StringType(),  True),
    StructField("region",         StringType(),  True),
    StructField("priority",       StringType(),  True),
    StructField("order_date",     StringType(),  True),
    StructField("quantity",       IntegerType(), True),
    StructField("payment_method", StringType(),  True),
    StructField("old_name",       StringType(),  True),
])

In [40]:
import shutil
for p in ["pipeline_parquet", "pipeline_csv"]:
    shutil.rmtree(p, ignore_errors=True)

pipeline = (spark.read.csv("../data/orders_data.csv", header=True, schema=schema)  # read
    .withColumnRenamed("old_name", "new_name")                             # transform: rename
    .withColumn("price", col("price").cast(DoubleType()))                  # transform: cast
    .withColumn("final_price", F.round(col("base_price") * 1.18, 2))       # transform: add col
    .filter(col("user_id").isNotNull() & (col("user_id") != ""))           # filter: valid users
    .filter(col("status") == "Completed"))                                 # filter: completed

print("Pipeline output rows:", pipeline.count())
pipeline.select("order_id", "new_name", "price", "final_price", "status").show(7)

# write: save processed data in both formats
pipeline.write.mode("overwrite").parquet("pipeline_parquet")
pipeline.write.mode("overwrite").csv("pipeline_csv", header=True)
print("Saved pipeline output to Parquet and CSV.")

Pipeline output rows: 1865
+---------+--------+-------+-----------+---------+
| order_id|new_name|  price|final_price|   status|
+---------+--------+-------+-----------+---------+
|ORD100003|item_245|3072.87|    1670.95|Completed|
|ORD100004|item_381|4424.95|    3400.36|Completed|
|ORD100006|item_422| 2778.0|    3125.88|Completed|
|ORD100007|item_427|4779.61|    3665.13|Completed|
|ORD100009|item_214|   NULL|     589.32|Completed|
|ORD100019|item_159|2476.34|    2500.01|Completed|
|ORD100026|item_132| 739.85|    3105.36|Completed|
+---------+--------+-------+-----------+---------+
only showing top 7 rows

Saved pipeline output to Parquet and CSV.


## 12. Best Practices for Large Datasets -> `show()` over `collect()`

**Q15 - Why `.show()` is safer than `.collect()` on a multi-terabyte dataset:**

- `.collect()` retrieves **all rows** from the executors and brings them to the **driver**. If the dataset is very large, this can consume excessive memory and may cause the application to fail with an **OutOfMemory** error.

- `.show()` returns only a small number of rows for preview. Spark processes only the required records, making it a safe and efficient option for inspecting data without loading the entire dataset into the driver's memory.


NOTE : use `show(n)`, `take(n)`, or `limit(n)` for exploration; only `collect()` when you are certain the result is small.

In [41]:
# Safe: returns only a few rows to the driver
df.select("order_id", "amount").show(5)

# take(n) is also safe — returns a small Python list
sample = df.take(3)
print("take(3) returned", len(sample), "rows to the driver (safe).")

+---------+-------+
| order_id| amount|
+---------+-------+
|ORD100000|2123.09|
|ORD100001|4088.83|
|ORD100002|2520.27|
|ORD100003|4873.15|
|ORD100004|1826.08|
+---------+-------+
only showing top 5 rows

take(3) returned 3 rows to the driver (safe).


## Brief Insights on Performance & Architecture

- **Spark follows a distributed architecture.** The Driver creates the execution plan and schedules tasks, the Cluster Manager provides resources, and the Executors process data in parallel across the cluster.

- **Lazy evaluation improves efficiency.** Transformations are recorded instead of executed immediately, allowing Spark to optimize the complete execution plan before running it.

- **Lineage provides fault tolerance.** Spark tracks the sequence of transformations, enabling it to recompute only the lost partitions if an executor fails rather than restarting the entire job.

- **Parquet is more efficient than CSV for analytics.** Its columnar storage, built-in schema, compression, and predicate pushdown reduce disk I/O and improve query performance.

- **Shuffles are expensive operations.** Wide transformations such as `groupBy()` and `join()` require data movement across partitions, making them one of the primary factors affecting Spark job performance.

- **Manage driver memory carefully.** Use `show()`, `take()`, or `limit()` to inspect data, and avoid `collect()` on large datasets unless the result is guaranteed to fit in the driver's memory.

In [ ]:
# Clean up temporary output directories created during the demo
import shutil
for p in ["out_csv","out_parquet","pipeline_parquet","pipeline_csv","q12_output_csv"]:
    shutil.rmtree(p, ignore_errors=True)

spark.stop()
print("Temp outputs cleaned and Spark session stopped.")

Temp outputs cleaned and Spark session stopped.
